In [ ]:
import pandas as pd
import numpy as np
import os
import gc
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
PATH_PARQUET   = '../05_Master_Data/Master_Headcount_Full.parquet'
PATH_OUTPUT    = '../06_Model_Data/'
PATH_PROMO_CSV = '../03_Raw_Promotion/promotion_history.csv'
os.makedirs(PATH_OUTPUT, exist_ok=True)

FORWARD_WINDOW_MONTHS = 3
EXPECTED_LVO_2025     = 1883
TRAIN_END             = pd.Timestamp('2023-12-31')
VAL_END               = pd.Timestamp('2024-12-31')

# ============================================================
# NEW JOINER THRESHOLD — single source of truth
# ============================================================
NEW_JOINER_MONTHS = 18   # < 18m = promo logic not applicable

NULL_STRINGS = {'NAN', 'NONE', 'NULL',
                'NAT', '', 'NA', 'N/A'}

# ============================================================
# MANAGEMENT LEVEL MAP
# ============================================================
MGMT_RAW_TO_ORDINAL = {
    'Contingent Worker':          0,
    'Intern':                     0,
    'Admin Business Coordinator': 1,
    'Admin Business Lead':        1,
    'Admin Business Partner':     1,
    'Administrative Assistant':   1,
    'Executive Assistant':        1,
    'Employee Fixed Term':        1,
    'Analyst':                    2,
    'Specialist':                 3,
    'Associate':                  4,
    'Senior Associate':           4,
    'Vice President':             5,
    'Director':                   6,
    'Principal':                  6,
    'Executive Director':         6,
    'Managing Director':          7,
    'Senior Managing Director':   7,
    'Fund Partner':               7,
    'Partner':                    7,
    'President':                  7,
    'CEO-Chairman':               7,
    'Vice Chairman':              7,
    'Senior Advisor':             7,
}

EXPECTED_PROMO_MONTHS_MAP = {
    1: 24,
    2: 30,
    3: 36,
    4: 48,
    5: 60,
    6: 72,
    7: 120,
}

# ============================================================
# COLUMN DROP LISTS
# ============================================================
DROP_ON_LOAD = [
    'HC_Future_Term_Current Year_EE_Invol',
    'HC_Future_Term_Current Year_EE_Other',
    'HC_Future_Term_EE_Current_Year_1_0',
    'HC_Future_Term_EE',
    'Worker', 'Candidate Name', 'Job Profile',
    'Location',
    'US Ethnicity', 'Generation',
    'Performance_Text', 'Intent_Stay_EOS',
    'Management Level As of Latest Hire Date',
    'Length of Service Tenure Category',
    'Level 1 Org', 'Level 10 Org',
    'Termination Reason',
    'HC_Did_Not_Start_Hire_EE',
    'HC_Did_Not_Start_Term_EE',
    'HC_START_INT_ASSI_EE', 'HC_END_INT_ASSI_EE',
    'WA_CHK_CTT_EE',
    'HC_CTT_LTD', 'HC_CTT_INT',
    'HC_CTT_CWKExOffshore', 'HC_CTT_CWKNonworker',
    'HC_CTT_CWKOffshore',
    'HC_DEC_CWKExOffshore', 'HC_DEC_CWKNonworker',
    'HC_DEC_CWKOffshore',
    'HC_DEC_EE_LTD', 'HC_DEC_INT',
    'HC_BSL_EE_Old', 'ADJ_HC_BSL_EE',
    'HC_CTT_Comm', 'HC_XOT_Comm', 'HC_XIN_Comm',
    'HC_XTT_Comm',
    'HC_RGI_Comm', 'HC_RGO_Comm', 'HC_RTT_Comm',
    'HC_Hire_Campus_Comm', 'HC_Hire_Lateral_Comm',
    'HC_ACQ_Comm', 'HC_HIR_Comm', 'HC_HTT_Comm',
    'HC_XTT_EE', 'HC_RTT_EE', 'HC_HTT_EE',
    'HC_ACQ_EE', 'HC_INT_to_ANLST',
    'Data as of Year', 'Latest Hire Month Number',
    'Intern Class Year',
]

POST_ENGINEER_DROP = [
    'HC_LVO_EE',
    'HC_Future_Term_Current Year_EE_Vol',
    'HC_LIN_EE', 'HC_LOT_EE', 'HC_LTT_EE',
    'HC_DLT_EE',
    'HC_XOT_EE', 'HC_XIN_EE', 'HC_RGO_EE',
    'HC_RGI_EE',
    'HC_BSL_EE', 'HC_DEC_EE', 'HC_HIR_EE',
    'Latest_Hire_Date_dt', 'Original_Hire_Date_dt',
    'Termination_Date_dt',
    'Latest Hire Date', 'Original Hire Date',
    'Termination Date', 'Snapshot_Date',
    'Termination_Date',
    'Continuous Service Date', 'Date of Birth',
    'Manager ID', 'Manager',
    'Position ID', 'Termination Reason Category',
    'Termination Month Number',
    'Company ID',
    'HC_CTT_EE',
    'Management Level',
    'Acquisition Company',
    'iHub Locations Flag',
    'International Assignment Flag',
    'CFA Flag', 'MBA Degree Name', 'PhD Degree Name',
    'LATAM Org Flag', 'GEC Committee Flag',
    'C20 Committee Flag',
    'Time Type', 'US Regulatory Flag',
    'Job Family', 'Country',
    'Analyst Class Year',
    'Length of Service', 'Time In Title (Current)',
    'HC_CTT_On_Leave', 'EOS_Score_Is_Actual',
    'Exit_Month', 'is_exit_month',
    'Performance_Score', 'Is_Promoted',
    # ── FIX: removed 'promo_fallback_used' from drop list.
    # It is now renamed to 'has_promo_history' and kept
    # in the model so trees can distinguish real mslp
    # from tenure-fallback mslp.
    # ALSO dropping the 4 redundant binary tenure flags
    # that are deterministic cuts of tenure_months and
    # inflate SHAP attribution without adding signal.
    'is_early_tenure',
    'is_honeymoon_cliff',
    'is_promotion_window',
    'is_long_tenure',
]


# ============================================================
# PROMOTION HISTORY LOAD
# ============================================================
print("=" * 60)
print("LOADING PROMOTION HISTORY CSV")
print("=" * 60)

if os.path.exists(PATH_PROMO_CSV):
    ph = pd.read_csv(
        PATH_PROMO_CSV,
        encoding='utf-8',
        skiprows=2,
        on_bad_lines='skip',
        low_memory=False
    )
    ph.columns = ph.columns.str.strip()
    ph['Employee_ID'] = (
        ph['Employee ID']
        .astype(str).str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )
    ph['Process_Year'] = pd.to_numeric(
        ph['Process Year'], errors='coerce')

    print(f"  Total rows        : {len(ph):,}")
    print(f"  Unique employees  : "
          f"{ph['Employee_ID'].nunique():,}")
    print(f"  Nomination Status - After Memo:")
    print(ph['Nomination Status - After Memo']
          .value_counts().to_string())

    ph_promoted = ph[
        ph['Nomination Status - After Memo']
        .str.strip() == 'Nominated'
    ].copy()

    print(f"\n  Nominated (promoted) : "
          f"{len(ph_promoted):,}")
    print(f"  Withdrawn / other    : "
          f"{len(ph) - len(ph_promoted):,}")
    print(f"  Unique promoted emps : "
          f"{ph_promoted['Employee_ID'].nunique():,}")

    latest_promo = (
        ph_promoted
        .dropna(subset=['Process_Year', 'Employee_ID'])
        .groupby('Employee_ID', as_index=False)
        ['Process_Year'].max()
        .rename(columns={
            'Process_Year': 'Last_Promo_Year'})
    )

    latest_promo['Last_Promo_Date'] = pd.to_datetime(
        (latest_promo['Last_Promo_Year'] + 1)
        .astype(int).astype(str) + '-01-01',
        format='%Y-%m-%d'
    )

    print(f"\n  Effective date rule:")
    print(f"  PY2025 -> Jan 1 2026")
    print(f"  PY2024 -> Jan 1 2025")
    print(f"  PY2023 -> Jan 1 2024")
    print(f"\n  Promo year distribution:")
    print(latest_promo['Last_Promo_Year']
          .value_counts().sort_index().to_string())

    PROMO_LOOKUP_AVAILABLE = True
    print(f"\n  Promotion lookup ready -- "
          f"{len(latest_promo):,} employees")
else:
    print(f"  Not found: {PATH_PROMO_CSV}")
    latest_promo           = None
    PROMO_LOOKUP_AVAILABLE = False


# ============================================================
# SECTION 1 -- LOAD & EXTRACT EXIT DATES
# ============================================================
print("\n" + "=" * 60)
print("SECTION 1 -- LOAD & EXTRACT EXIT DATES")
print("=" * 60)

print("Step 1a: Building rehire-aware exit lookup...")
df_exit_raw = pd.read_parquet(
    PATH_PARQUET,
    columns=[
        'Employee_ID', 'Month_Date', 'HC_LVO_EE',
        'Termination Reason Category',
        'Termination Date'
    ]
)
df_exit_raw['Month_Date'] = pd.to_datetime(
    df_exit_raw['Month_Date'])
df_exit_raw['Termination_Date'] = pd.to_datetime(
    df_exit_raw['Termination Date'], errors='coerce')

print(f"  Mini-load shape : {df_exit_raw.shape}")
print(f"  HC_LVO_EE value counts:")
print(df_exit_raw['HC_LVO_EE']
      .value_counts().to_string())

lvo_vol = df_exit_raw[
    (df_exit_raw['HC_LVO_EE'] == -1) &
    (df_exit_raw['Termination Reason Category']
     == 'Voluntary')
].copy()

print(f"\n  Termination reason breakdown "
      f"for HC_LVO_EE==-1:")
print(
    df_exit_raw[df_exit_raw['HC_LVO_EE'] == -1]
    ['Termination Reason Category']
    .value_counts().to_string()
)
print(f"\n  After Voluntary filter:")
print(f"  Rows            : {len(lvo_vol):,}")
print(f"  Unique employees: "
      f"{lvo_vol['Employee_ID'].nunique():,}")

exit_events = (
    lvo_vol
    .sort_values(['Employee_ID', 'Month_Date'])
    .groupby(
        ['Employee_ID', 'Termination_Date'],
        as_index=False)
    .agg(
        Exit_Month  =('Month_Date', 'min'),
        Term_Reason =(
            'Termination Reason Category', 'first')
    )
)

print(f"\n  Total distinct exit events    : "
      f"{len(exit_events):,}")
print(f"  Employees with multiple exits : "
      f"{(exit_events.groupby('Employee_ID').size() > 1).sum():,}")

latest_exit = (
    exit_events
    .sort_values('Termination_Date')
    .groupby('Employee_ID', as_index=False)
    .last()
    .rename(columns={'Exit_Month': 'Exit_Month_Raw'})
)
latest_exit['Exit_Month'] = (
    latest_exit['Exit_Month_Raw'] +
    pd.offsets.MonthEnd(0)
)
latest_exit.drop(
    columns=['Exit_Month_Raw', 'Term_Reason'],
    inplace=True)

print(f"\n  Final unique employees in lookup: "
      f"{len(latest_exit):,}")
print(f"\n  LVO by snapshot year (rehire-aware):")
print(
    latest_exit
    .groupby(latest_exit['Exit_Month'].dt.year)
    ['Employee_ID'].nunique().to_string()
)

lvo_2025_raw = latest_exit[
    latest_exit['Exit_Month'].dt.year == 2025
]['Employee_ID'].nunique()
print(f"\n  LVO 2025 in lookup   : {lvo_2025_raw:,}")
print(f"  LVO 2025 People Pack : {EXPECTED_LVO_2025:,}")
print(f"  Difference           : "
      f"{lvo_2025_raw - EXPECTED_LVO_2025:+,}")
if lvo_2025_raw == EXPECTED_LVO_2025:
    print("  Exit lookup reconciles exactly")
else:
    print("  Small gap -- see documentation")

del df_exit_raw, lvo_vol, exit_events
gc.collect()

print("\nStep 1d: Loading full parquet...")
df = pd.read_parquet(PATH_PARQUET)

drop_existing = [
    c for c in DROP_ON_LOAD if c in df.columns]
df.drop(columns=drop_existing, inplace=True)

df = df[df['HC_CTT_EE'] == 1].reset_index(drop=True)

df['Month_Date'] = (
    pd.to_datetime(df['Month_Date']) +
    pd.offsets.MonthEnd(0)
)
df['Latest_Hire_Date_dt'] = pd.to_datetime(
    df['Latest Hire Date'], errors='coerce')
df['Original_Hire_Date_dt'] = pd.to_datetime(
    df['Original Hire Date'], errors='coerce')
df['Termination_Date_dt'] = pd.to_datetime(
    df['Termination Date'], errors='coerce')
df.drop(columns=[
    'Latest Hire Date', 'Original Hire Date',
    'Termination Date', 'Snapshot_Date'
], inplace=True, errors='ignore')

print(f"  Shape after filter & early drop : {df.shape}")
print(f"  RAM usage                        : "
      f"{df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"  Unique employees                 : "
      f"{df['Employee_ID'].nunique():,}")


# ============================================================
# SECTION 2 -- LABEL CONSTRUCTION
# ============================================================
print("\n" + "=" * 60)
print("SECTION 2 -- LABEL CONSTRUCTION")
print("=" * 60)

df.sort_values(['Employee_ID', 'Month_Date'],
               inplace=True)
df.reset_index(drop=True, inplace=True)

df = df.merge(latest_exit, on='Employee_ID',
              how='left')
matched = df[df['Exit_Month'].notna()][
    'Employee_ID'].nunique()
print(f"Employees matched to exit lookup : {matched:,}")

before = len(df)
df = df[
    df['Exit_Month'].isna() |
    (df['Month_Date'] <= df['Exit_Month'])
].reset_index(drop=True)
after = len(df)
print(f"Rows before phantom removal : {before:,}")
print(f"Rows after phantom removal  : {after:,}")
print(f"Phantom rows dropped        : "
      f"{before - after:,}")

df['is_exit_month'] = (
    df['Exit_Month'].notna() &
    (df['Month_Date'].dt.to_period('M') ==
     df['Exit_Month'].dt.to_period('M'))
).astype(np.int8)
print(f"Exit month rows marked           : "
      f"{df['is_exit_month'].sum():,}")
print(f"Unique employees with exit month : "
      f"{df[df['is_exit_month']==1]['Employee_ID'].nunique():,}")

print("\n-- RECONCILIATION ----------------------------------------")
_T_plus_3   = (df['Month_Date'] +
               pd.DateOffset(months=FORWARD_WINDOW_MONTHS))
_last_month = df['Month_Date'].max()
_cutoff     = (_last_month -
               pd.DateOffset(months=FORWARD_WINDOW_MONTHS))

_conditions = [
    df['Month_Date'] > _cutoff,
    (
        df['Exit_Month'].notna() &
        (df['Exit_Month'] > df['Month_Date']) &
        (df['Exit_Month'] <= _T_plus_3)
    ),
]
df['_temp_label'] = np.select(
    _conditions, [np.nan, 1], default=0)

lvo_2025_model = df[
    (df['_temp_label'] == 1) &
    (df['Exit_Month'].dt.year == 2025)
]['Employee_ID'].nunique()
df.drop(columns=['_temp_label'], inplace=True)

print(f"LVO 2025 -- Model      : {lvo_2025_model:,}")
print(f"LVO 2025 -- People Pack: {EXPECTED_LVO_2025:,}")
print(f"Difference             : "
      f"{lvo_2025_model - EXPECTED_LVO_2025:+,}")
if lvo_2025_model == EXPECTED_LVO_2025:
    print("RECONCILIATION PASSED -- exact match")
elif abs(lvo_2025_model - EXPECTED_LVO_2025) <= 5:
    print("RECONCILIATION CLOSE -- within 5")
else:
    print("RECONCILIATION FAILED -- investigate")

last_month = df['Month_Date'].max()
cutoff     = (last_month -
              pd.DateOffset(months=FORWARD_WINDOW_MONTHS))
T_plus_3   = (df['Month_Date'] +
              pd.DateOffset(months=FORWARD_WINDOW_MONTHS))

print(f"\nLast month in data    : "
      f"{last_month.strftime('%Y-%m')}")
print(f"Last labelable month  : "
      f"{cutoff.strftime('%Y-%m')}")

conditions = [
    df['Month_Date'] > cutoff,
    (
        df['Exit_Month'].notna() &
        (df['Exit_Month'] > df['Month_Date']) &
        (df['Exit_Month'] <= T_plus_3)
    ),
]
choices = [np.nan, 1]

df['attrition_label'] = np.select(
    conditions, choices, default=0
).astype(float)

df_live     = df[df['attrition_label'].isna()].copy()
df_labelled = df[df['attrition_label'].notna()].copy()
df_labelled['attrition_label'] = (
    df_labelled['attrition_label'].astype(np.int8))

del df
gc.collect()

print(f"\nLabelled rows : {len(df_labelled):,}")
print(f"Live rows     : {len(df_live):,}")
print(f"\nLabel distribution:")
print(df_labelled['attrition_label']
      .value_counts().to_string())
print(f"Positive rate : "
      f"{df_labelled['attrition_label'].mean()*100:.3f}%")
print(f"\nLabel=1 unique leavers by year:")
print(
    df_labelled[df_labelled['attrition_label'] == 1]
    .groupby(df_labelled['Month_Date'].dt.year)
    ['Employee_ID'].nunique().to_string()
)


# ============================================================
# SECTION 2B -- PRE-COMPUTE MANAGER FEATURES
# ============================================================
print("\n" + "=" * 60)
print("SECTION 2B -- PRE-COMPUTE MANAGER FEATURES")
print("=" * 60)

print("  Loading manager history (full parquet)...")
df_mgr = pd.read_parquet(
    PATH_PARQUET,
    columns=['Employee_ID', 'Month_Date',
             'Manager ID', 'HC_CTT_EE']
)
df_mgr['Month_Date'] = (
    pd.to_datetime(df_mgr['Month_Date']) +
    pd.offsets.MonthEnd(0)
)
df_mgr['Employee_ID'] = (
    df_mgr['Employee_ID']
    .astype(str).str.strip()
)
df_mgr = df_mgr[
    df_mgr['HC_CTT_EE'] == 1
].copy()

df_mgr.sort_values(
    ['Employee_ID', 'Month_Date'],
    inplace=True)
df_mgr.reset_index(drop=True, inplace=True)

print(f"  Rows loaded      : {len(df_mgr):,}")
print(f"  Unique employees : "
      f"{df_mgr['Employee_ID'].nunique():,}")

df_mgr['_mgr_raw'] = (
    df_mgr['Manager ID']
    .astype(str).str.strip()
    .str.replace(r'\.0$', '', regex=True)
    .str.upper().str.strip()
)
df_mgr['_mgr_clean'] = np.where(
    df_mgr['_mgr_raw'].isin(NULL_STRINGS),
    np.nan,
    df_mgr['_mgr_raw']
)
df_mgr['_mgr_ffill'] = (
    df_mgr.groupby('Employee_ID')['_mgr_clean']
    .transform(lambda x: x.ffill())
)
df_mgr['_mgr_prev'] = (
    df_mgr.groupby('Employee_ID')['_mgr_ffill']
    .shift(1)
)
df_mgr['_is_first'] = (
    df_mgr.groupby('Employee_ID').cumcount() == 0
)
df_mgr['_mgr_event'] = (
    (~df_mgr['_is_first']) &
    (df_mgr['_mgr_ffill'].notna()) &
    (df_mgr['_mgr_prev'].notna()) &
    (df_mgr['_mgr_ffill'] != df_mgr['_mgr_prev'])
).astype(np.int8)

df_mgr['manager_changes_24m'] = (
    df_mgr.groupby('Employee_ID')['_mgr_event']
    .transform(
        lambda x: x.rolling(
            window=24, min_periods=1
        ).sum()
    )
).fillna(0).astype(np.int8)

df_mgr['_mgr_cumsum'] = (
    df_mgr.groupby('Employee_ID')['_mgr_event']
    .cumsum()
)
df_mgr['current_manager_tenure_months'] = (
    df_mgr.groupby(
        ['Employee_ID', '_mgr_cumsum'])
    .cumcount()
    .astype(np.int8)
)

mgr_features = df_mgr[[
    'Employee_ID',
    'Month_Date',
    'manager_changes_24m',
    'current_manager_tenure_months'
]].copy()

n_changed = (
    df_mgr.groupby('Employee_ID')
    ['manager_changes_24m'].last() > 0
).sum()
print(f"\n  Employees with >=1 change : {n_changed:,}")
print(f"  manager_changes_24m dist :")
print(
    df_mgr.groupby('Employee_ID')
    ['manager_changes_24m'].last()
    .value_counts().sort_index().head(8)
    .to_string()
)

spot = df_mgr[
    df_mgr['Employee_ID'] == '483166'
][['Month_Date', '_mgr_ffill', '_mgr_prev',
   '_mgr_event', 'manager_changes_24m',
   'current_manager_tenure_months']].tail(12)
if len(spot) > 0:
    print(f"\n  Spot check Employee 483166:")
    print(spot.to_string())
    print(f"\n  Latest manager_changes_24m : "
          f"{spot['manager_changes_24m'].iloc[-1]}"
          f"  <- expect 2")
else:
    print("  Employee 483166 not found")

del df_mgr
gc.collect()

print(f"\n  Manager features pre-computed -- "
      f"{len(mgr_features):,} rows")


# ============================================================
# SECTION 3 -- FEATURE ENGINEERING
# ============================================================
print("\n" + "=" * 60)
print("SECTION 3 -- FEATURE ENGINEERING")
print("=" * 60)

def engineer_features_vectorised(df, dataset_name=''):
    df = df.copy()
    df.sort_values(['Employee_ID', 'Month_Date'],
                   inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"  Engineering: {dataset_name} "
          f"({len(df):,} rows)...")

    # ── [1] Tenure & Stability ───────────────────────────
    # FIX: Removed 4 redundant binary tenure flags:
    #   is_early_tenure, is_honeymoon_cliff,
    #   is_promotion_window, is_long_tenure.
    # These are all deterministic cuts of tenure_months.
    # Tree models already find these splits themselves.
    # Keeping them inflates SHAP attribution for the
    # tenure family without adding predictive signal.
    # We retain tenure_months (continuous), is_new_joiner
    # (guardrail anchor) and early_tenure_band (contextual
    # bucketing for NJ differentiation) only.
    print("  [1/10] Tenure & stability...")
    df['tenure_months'] = (
        df['Length of Service'] * 12).round(1)
    df['time_in_title_months'] = (
        df['Time In Title (Current)'] * 12).round(1)
    df['tenure_stability_ratio'] = (
        df['time_in_title_months'] /
        df['tenure_months'].replace(0, np.nan)
    ).clip(0, 1).fillna(0).round(4)

    # ── is_new_joiner: single source of truth ────────────
    df['is_new_joiner'] = (
        df['tenure_months'] < NEW_JOINER_MONTHS
    ).astype(np.int8)
    print(f"    is_new_joiner count : "
          f"{df['is_new_joiner'].sum():,}"
          f" (tenure < {NEW_JOINER_MONTHS}m)")

    # ── early_tenure_band: contextual NJ bucketing ───────
    df['early_tenure_band'] = pd.cut(
        df['tenure_months'],
        bins=[0, 6, 12, 18, 36, 60, 9999],
        labels=[0, 1, 2, 3, 4, 5],
        right=False
    ).astype(float).fillna(5).astype(np.int8)

    # ── [2] Rehire & Acquisition ─────────────────────────
    print("  [2/10] Rehire & acquisition...")
    df['is_rehire'] = (
        df['Latest_Hire_Date_dt'].notna() &
        df['Original_Hire_Date_dt'].notna() &
        (
            (df['Latest_Hire_Date_dt'] -
             df['Original_Hire_Date_dt'])
            .dt.days > 180
        )
    ).astype(np.int8)
    df['Is_Acquisition_Hire'] = (
        df['Acquisition Company']
        .astype(str).str.strip() != ''
    ).astype(np.int8)

    # ── [3] Management Level ─────────────────────────────
    print("  [3/10] Management level...")
    df['mgmt_ordinal'] = (
        df['Management Level']
        .map(MGMT_RAW_TO_ORDINAL)
        .fillna(1)
        .astype(np.int8)
    )

    # ── [4] Campus Hire ──────────────────────────────────
    print("  [4/10] Campus hire...")
    df['Is_Campus_Hire'] = (
        df['Analyst Class Year'] != 0
    ).astype(np.int8)
    df['Campus_Cohort_Age'] = np.where(
        df['Is_Campus_Hire'] == 1,
        df['Year'] - df['Analyst Class Year'],
        -1
    ).astype(np.int8)

    # ── [5] Manager Stability ────────────────────────────
    print("  [5/10] Manager stability "
          "(merge from pre-compute)...")
    df = df.merge(
        mgr_features,
        on=['Employee_ID', 'Month_Date'],
        how='left'
    )
    df['manager_changes_24m'] = (
        df['manager_changes_24m']
        .fillna(0).astype(np.int8)
    )
    df['current_manager_tenure_months'] = (
        df['current_manager_tenure_months']
        .fillna(0).astype(np.int8)
    )
    spot = df[
        df['Employee_ID'].astype(str) == '483166'
    ]['manager_changes_24m']
    if len(spot) > 0:
        print(f"    Spot check 483166 : "
              f"{spot.iloc[-1]}  <- expect 2")

    # ── [6] Promotion Features ───────────────────────────
    # FIX: Two changes here vs previous version:
    #
    # (1) promo_fallback_used is now renamed to
    #     has_promo_history and KEPT in the model.
    #     Previously it was dropped, making mslp
    #     ambiguous -- the model could not distinguish
    #     "genuinely 36m since last promotion" from
    #     "joined 36m ago, no CSV record exists".
    #     With has_promo_history in the feature set,
    #     trees can split: (mslp high AND has_history=1)
    #     vs (mslp high AND has_history=0).
    #
    # (2) Fallback employees (no promo CSV record,
    #     tenure >= 18m) now receive mslp = NaN instead
    #     of mslp = tenure_months.
    #     Previously mslp was a literal copy of tenure
    #     for these employees, creating a hidden
    #     collinear duplicate.
    #     NaN is imputed via SimpleImputer (median of
    #     employees who DO have promo history) in NB03.
    #     This breaks the tenure = mslp identity for
    #     the fallback population.
    print("  [6/10] Promotion features "
          "(history CSV + NaN fallback + "
          "new joiner fix)...")

    if PROMO_LOOKUP_AVAILABLE and \
            latest_promo is not None:
        df['_emp_str'] = (
            df['Employee_ID']
            .astype(str).str.strip()
            .str.replace(r'\.0$', '', regex=True)
        )
        _lp = latest_promo.copy()
        _lp['_emp_str'] = (
            _lp['Employee_ID']
            .astype(str).str.strip()
            .str.replace(r'\.0$', '', regex=True)
        )
        df = df.merge(
            _lp[['_emp_str', 'Last_Promo_Date']],
            on='_emp_str',
            how='left'
        )
        df.drop(columns=['_emp_str'],
                inplace=True, errors='ignore')
        n_matched = df['Last_Promo_Date'].notna().sum()
        n_emp_matched = df[
            df['Last_Promo_Date'].notna()
        ]['Employee_ID'].nunique()
        print(f"    Source 1 -- history CSV: "
              f"{n_emp_matched:,} employees "
              f"({n_matched:,} rows)")
    else:
        df['Last_Promo_Date'] = pd.NaT

    # ── FIX (1): has_promo_history kept in model ─────────
    # True  = employee has a record in the promo CSV
    # False = no record; mslp will be NaN (imputed later)
    df['has_promo_history'] = (
        df['Last_Promo_Date'].notna()
    ).astype(np.int8)

    n_with_history    = df['has_promo_history'].sum()
    n_without_history = len(df) - n_with_history
    print(f"    has_promo_history=1 : "
          f"{n_with_history:,} rows")
    print(f"    has_promo_history=0 : "
          f"{n_without_history:,} rows "
          f"(mslp will be NaN -> imputed)")

    has_promo = df['Last_Promo_Date'].notna()
    no_promo  = ~has_promo

    # ── Compute mslp for employees WITH promo history ────
    df['months_since_last_promotion'] = np.nan
    df.loc[has_promo,
           'months_since_last_promotion'] = (
        (df.loc[has_promo, 'Month_Date'] -
         df.loc[has_promo, 'Last_Promo_Date'])
        .dt.days / 30.44
    ).round(0)

    # ── FIX (2): NJ with no promo history get 0 ──────────
    # New joiners genuinely haven't had time to be
    # promoted -- 0 is correct, not NaN.
    new_joiner_mask = (df['is_new_joiner'] == 1)
    fallback_nj     = no_promo & new_joiner_mask

    df.loc[fallback_nj,
           'months_since_last_promotion'] = 0

    # ── Non-NJ without history: leave as NaN ─────────────
    # SimpleImputer in NB03 fills with median of
    # employees who DO have promo history.
    # This prevents the tenure = mslp identity.
    # (no action needed -- already NaN)
    n_nan_fallback = (
        no_promo & ~new_joiner_mask &
        df['months_since_last_promotion'].isna()
    ).sum()
    print(f"    Non-NJ no-history rows (NaN->imputed): "
          f"{n_nan_fallback:,}")

    # ── Clip only the real promo history values ───────────
    # Do NOT fillna(0) here -- NaN must reach imputer.
    df['months_since_last_promotion'] = (
        df['months_since_last_promotion']
        .clip(upper=120)
    )
    # Cast to float to preserve NaN (int16 can't hold NaN)
    df['months_since_last_promotion'] = (
        df['months_since_last_promotion']
        .astype('float32')
    )

    df.drop(columns=['Last_Promo_Date'],
            inplace=True, errors='ignore')

    # ── promotion_gap_months ──────────────────────────────
    df['_expected_promo'] = (
        df['mgmt_ordinal']
        .map(EXPECTED_PROMO_MONTHS_MAP)
        .fillna(36).clip(upper=120)
    )
    # gap is NaN where mslp is NaN -- imputer handles it
    df['promotion_gap_months'] = (
        df['months_since_last_promotion'] -
        df['_expected_promo']
    )

    # MD+ never has a gap
    df.loc[df['mgmt_ordinal'] == 7,
           'promotion_gap_months'] = 0.0

    # Neutralise gap for new joiners
    df.loc[new_joiner_mask,
           'promotion_gap_months'] = 0.0

    # Clamp gap >= 0 for everyone (no business meaning
    # to "negative overdue")
    df['promotion_gap_months'] = (
        df['promotion_gap_months']
        .clip(lower=0, upper=120)
        .astype('float32')
    )

    df.drop(columns=['_expected_promo'],
            inplace=True, errors='ignore')

    # ── Promotion validation ─────────────────────────────
    n_neg_gap = (df['promotion_gap_months'] < 0).sum()
    n_nan_mslp = df[
        'months_since_last_promotion'].isna().sum()
    n_nan_gap  = df['promotion_gap_months'].isna().sum()

    print(f"\n    -- Promotion Validation --")
    print(f"    Negative gaps        : "
          f"{n_neg_gap}  <- must be 0")
    print(f"    NaN mslp (-> imputer): "
          f"{n_nan_mslp:,}")
    print(f"    NaN gap  (-> imputer): "
          f"{n_nan_gap:,}")

    nj_mslp = df.loc[
        new_joiner_mask,
        'months_since_last_promotion'].describe()
    nj_gap = df.loc[
        new_joiner_mask,
        'promotion_gap_months'].describe()
    print(f"\n    New joiner mslp stats:")
    print(nj_mslp.round(1).to_string())
    print(f"\n    New joiner gap stats:")
    print(nj_gap.round(1).to_string())

    assert n_neg_gap == 0, (
        "FATAL: negative promotion gaps found "
        "after fix -- investigate!")
    print(f"\n    Negative gap assert   : PASSED")

    promo_cov = df['has_promo_history'].mean() * 100
    print(f"    Promo CSV coverage    : {promo_cov:.1f}%")
    print(f"    mslp distribution (employees with history):")
    print(df.loc[has_promo,
                 'months_since_last_promotion']
          .describe().round(1).to_string())

    # ── [6b] Snapshot year ───────────────────────────────
    df['snapshot_year'] = (
        df['Month_Date'].dt.year.astype(np.int16))

    # ── [6c] Performance delta ───────────────────────────
    print("  [6c/10] Performance delta...")
    if 'Performance_Score' in df.columns:
        perf_annual = (
            df[['Employee_ID', 'snapshot_year',
                'Performance_Score']]
            .dropna(subset=['Performance_Score'])
            .groupby(
                ['Employee_ID', 'snapshot_year'],
                as_index=False)
            ['Performance_Score'].max()
            .rename(columns={
                'Performance_Score':
                    '_perf_this_year'
            })
        )
        perf_annual = perf_annual.sort_values(
            ['Employee_ID', 'snapshot_year'])
        perf_annual['_perf_prior_year'] = (
            perf_annual
            .groupby('Employee_ID')
            ['_perf_this_year'].shift(1)
        )
        perf_annual['performance_delta'] = (
            perf_annual['_perf_this_year'] -
            perf_annual['_perf_prior_year']
        ).astype(float)
        df = df.merge(
            perf_annual[[
                'Employee_ID', 'snapshot_year',
                'performance_delta']],
            on=['Employee_ID', 'snapshot_year'],
            how='left'
        )
        df['perf_improved'] = (
            df['performance_delta'] > 0
        ).astype(np.int8)
        df['perf_declined'] = (
            df['performance_delta'] < 0
        ).astype(np.int8)
        if 'EOS_Score_Delta' in df.columns:
            df['eos_and_perf_both_declined'] = (
                (df['perf_declined'] == 1) &
                (df['EOS_Score_Delta'] < 0)
            ).astype(np.int8)
        else:
            df['eos_and_perf_both_declined'] = (
                np.int8(0))
        print(f"    Perf coverage  : "
              f"{df['Performance_Score'].notna().mean()*100:.1f}%")
        print(f"    Delta null     : "
              f"{df['performance_delta'].isna().sum():,}")
        print(f"    perf_improved  : "
              f"{df['perf_improved'].sum():,}")
        print(f"    perf_declined  : "
              f"{df['perf_declined'].sum():,}")
        print(f"    eos+perf both down : "
              f"{df['eos_and_perf_both_declined'].sum():,}")
    else:
        print("    Performance_Score not found")
        df['performance_delta']          = np.nan
        df['perf_improved']              = np.int8(0)
        df['perf_declined']              = np.int8(0)
        df['eos_and_perf_both_declined'] = np.int8(0)

    # ── [7] Seasonality & Market Context ─────────────────
    print("  [7/10] Seasonality & market context...")
    df['snapshot_month'] = (
        df['Month_Date'].dt.month.astype(np.int8))
    df['is_bonus_season'] = (
        df['snapshot_month'].isin([2, 3])
    ).astype(np.int8)
    df['is_pre_bonus'] = (
        df['snapshot_month'] == 1
    ).astype(np.int8)
    df['great_resignation'] = (
        (df['Month_Date'] >= '2021-10-31') &
        (df['Month_Date'] <= '2022-06-30')
    ).astype(np.int8)

    # ── [8] Binary Flag Encoding ─────────────────────────
    print("  [8/10] Binary flags...")
    df['iHub_flag'] = (
        df['iHub Locations Flag'] == 'Yes'
    ).astype(np.int8)
    df['intl_assignment_flag'] = (
        df['International Assignment Flag'] == 'Yes'
    ).astype(np.int8)
    df['cfa_flag'] = (
        df['CFA Flag'] == 'Yes'
    ).astype(np.int8)
    df['mba_flag'] = (
        df['MBA Degree Name'] != ''
    ).astype(np.int8)
    df['phd_flag'] = (
        df['PhD Degree Name'] != ''
    ).astype(np.int8)
    df['on_leave_flag'] = (
        df['HC_CTT_On_Leave']
    ).astype(np.int8)
    df['latam_flag'] = (
        df['LATAM Org Flag'] == 'Yes'
    ).astype(np.int8)
    df['gec_flag'] = (
        df['GEC Committee Flag'] == 'Yes'
    ).astype(np.int8)
    df['c20_flag'] = (
        df['C20 Committee Flag'] == 'Yes'
    ).astype(np.int8)
    df['full_time_flag'] = (
        df['Time Type'] == 'Full time'
    ).astype(np.int8)
    df['us_regulatory_flag'] = (
        df['US Regulatory Flag'] == 'Yes'
    ).astype(np.int8)
    df['gender_encoded'] = (
        df['Gender']
        .map({'Male': 0, 'Female': 1,
              'I Prefer Not To Answer': 2,
              '': -1})
        .fillna(-1).astype(np.int8)
    )

    # ── [9] Grouped Categoricals ─────────────────────────
    print("  [9/10] Grouped categoricals...")
    top_countries = (
        df['Country'].value_counts()
        .nlargest(10).index)
    df['country_grouped'] = df['Country'].where(
        df['Country'].isin(top_countries),
        other='Other')
    jf_threshold = len(df) * 0.01
    small_jf = df['Job Family'].value_counts()
    small_jf = small_jf[
        small_jf < jf_threshold].index
    df['job_family_grouped'] = df[
        'Job Family'].where(
        ~df['Job Family'].isin(small_jf),
        other='Other')

    # ── [10] New joiner diagnostic ───────────────────────
    print("  [10/10] New joiner diagnostic...")
    nj_count = int(df['is_new_joiner'].sum())
    nj_pct   = float(nj_count / max(len(df), 1) * 100)
    print(f"    New joiners (<{NEW_JOINER_MONTHS}m) : "
          f"{nj_count:,} ({nj_pct:.1f}%)")
    print(f"    mslp=0 for new joiners   : "
          f"{(df.loc[df['is_new_joiner']==1, 'months_since_last_promotion']==0).sum():,}")
    print(f"    gap=0 for new joiners    : "
          f"{(df.loc[df['is_new_joiner']==1, 'promotion_gap_months']==0).sum():,}")
    print(f"    Any negative gaps        : "
          f"{(df['promotion_gap_months']<0).sum()}"
          f"  <- must be 0")

    print(f"\n  Feature engineering complete: "
          f"{dataset_name}")
    return df


df_labelled = engineer_features_vectorised(
    df_labelled, 'labelled set')
df_live     = engineer_features_vectorised(
    df_live,     'live set')


# ============================================================
# SECTION 4 -- FINAL COLUMN SELECTION
# ============================================================
print("\n" + "=" * 60)
print("SECTION 4 -- FINAL COLUMN SELECTION")
print("=" * 60)

for df_split in [df_labelled, df_live]:
    to_drop = [
        c for c in POST_ENGINEER_DROP
        if c in df_split.columns]
    df_split.drop(
        columns=to_drop, inplace=True,
        errors='ignore')

id_target    = [
    'Employee_ID', 'Month_Date', 'Year',
    'attrition_label']
feature_cols = [
    c for c in df_labelled.columns
    if c not in id_target]

print(f"Final labelled shape : {df_labelled.shape}")
print(f"Final live shape     : {df_live.shape}")
print(f"Total RAM usage      : "
      f"{df_labelled.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"\nFeature columns ({len(feature_cols)}):")
print(f"{'Column':<48} {'DType':<12} Nulls%")
print("-" * 72)
for c in sorted(feature_cols):
    dtype    = str(df_labelled[c].dtype)
    null_pct = df_labelled[c].isna().mean() * 100
    print(f"  {c:<46} {dtype:<12} {null_pct:.1f}%")

# ── CHANGE SUMMARY ────────────────────────────────────────
print(f"\n  *** CHANGES FROM PREVIOUS VERSION ***")
print(f"  Removed from model (redundant tenure flags):")
print(f"    is_early_tenure, is_honeymoon_cliff,")
print(f"    is_promotion_window, is_long_tenure")
print(f"  Added to model:")
print(f"    has_promo_history (replaces dropped promo_fallback_used)")
print(f"  Changed:")
print(f"    mslp fallback: tenure_months -> NaN for non-NJ no-history employees")
print(f"    mslp dtype: int16 -> float32 (to preserve NaN for imputer)")
print(f"    promotion_gap dtype: int16 -> float32 (same reason)")

# ── New joiner summary ────────────────────────────────────
print(f"\n  New joiner summary across splits:")
for nm, sp in [
    ('labelled', df_labelled),
    ('live',     df_live)
]:
    if 'is_new_joiner' in sp.columns:
        nj = int(sp['is_new_joiner'].sum())
        pct = float(nj / max(len(sp), 1) * 100)
        print(f"    {nm:<10}: {nj:,} "
              f"({pct:.1f}%) new joiners")


# ============================================================
# SECTION 5 -- TEMPORAL SPLIT & SAVE
# ============================================================
print("\n" + "=" * 60)
print("SECTION 5 -- TEMPORAL SPLIT & SAVE")
print("=" * 60)

df_train = df_labelled[
    df_labelled['Month_Date'] <= TRAIN_END
].copy()
df_val   = df_labelled[
    (df_labelled['Month_Date'] > TRAIN_END) &
    (df_labelled['Month_Date'] <= VAL_END)
].copy()
df_test  = df_labelled[
    df_labelled['Month_Date'] > VAL_END
].copy()

print(f"\n{'Split':<6} | {'Rows':>9} | "
      f"{'Employees':>10} | {'Leavers':>8} | "
      f"{'Pos Rate':>10} | {'New Joiners':>12}")
print("-" * 70)
for name, split in [
    ('TRAIN', df_train), ('VAL', df_val),
    ('TEST',  df_test),  ('LIVE', df_live),
]:
    n   = len(split)
    emp = split['Employee_ID'].nunique()
    nj  = int(split['is_new_joiner'].sum()) \
        if 'is_new_joiner' in split.columns else 0
    if ('attrition_label' in split.columns and
            split['attrition_label'].notna().any()):
        lvo = int(split['attrition_label'].sum())
        pct = float(
            split['attrition_label'].mean() * 100)
        print(f"{name:<6} | {n:>9,} | {emp:>10,} | "
              f"{lvo:>8,} | {pct:>9.3f}% | "
              f"{nj:>12,}")
    else:
        print(f"{name:<6} | {n:>9,} | {emp:>10,} | "
              f"{'N/A':>8} | {'N/A':>10} | "
              f"{nj:>12,}")

df_train.to_parquet(
    os.path.join(PATH_OUTPUT, 'abt_train.parquet'),
    index=False)
df_val.to_parquet(
    os.path.join(PATH_OUTPUT, 'abt_val.parquet'),
    index=False)
df_test.to_parquet(
    os.path.join(PATH_OUTPUT, 'abt_test.parquet'),
    index=False)
df_live.to_parquet(
    os.path.join(PATH_OUTPUT, 'abt_live.parquet'),
    index=False)

print(f"\nAll 4 splits saved to: {PATH_OUTPUT}")
print("   abt_train.parquet  -- training set")
print("   abt_val.parquet    -- validation")
print("   abt_test.parquet   -- holdout test")
print("   abt_live.parquet   -- live scoring")
print("\n" + "=" * 60)
print("ABT PIPELINE COMPLETE")
print("=" * 60)
